In [ ]:
# ==============================================================================
# STAGE 2: SUPERVISED FINE-TUNING (SFT) WITH COMPLIANT PROMPT STRUCTURING
# ==============================================================================
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
BASE_DIR = "/content/drive/MyDrive/domain-ai-assistant-finetuning"
MODEL_DIR = os.path.join(BASE_DIR, "models")
DATA_DIR = os.path.join(BASE_DIR, "data")

print("🔄 Step 1: Loading base model weights for instruction training...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
EOS_TOKEN = tokenizer.eos_token

# ------------------------------------------------------------------------------
# Step 2: Fixed Dataset Preprocessing Function
# ------------------------------------------------------------------------------
def format_prompts(examples):
    prompts = examples["prompt"]
    completions = examples["completion"]
    texts = []

    for prompt, completion in zip(prompts, completions):
        text = (
            f"<|im_start|>system\nYou are an expert customer support assistant. Provide clear, accurate, and structured answers.<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n{completion}{EOS_TOKEN}"
        )
        texts.append(text)

    return {"text": texts}

print("📊 Step 2: Executing text format mapping transformations...")
dataset_path = os.path.join(DATA_DIR, "instruction_dataset.jsonl")
raw_dataset = load_dataset("json", data_files={"train": dataset_path})
formatted_dataset = raw_dataset.map(format_prompts, batched=True)

# ------------------------------------------------------------------------------
# Step 3: Run the Training Loop (PICKLE BUG FIXED HERE)
# ------------------------------------------------------------------------------
print("🛠️ Step 3: Setting up the SFT training configuration...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 15,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        # FIXED BELOW: Bypasses intermediate serialization to fix PicklingError
        save_strategy = "no",
        gradient_checkpointing = False
    ),
)

print("🔥 Step 4: Initiating Supervised Fine-Tuning optimization phase...")
trainer.train()

print("💾 Step 5: Archiving SFT adapter layers...")
sft_adapter_path = os.path.join(MODEL_DIR, "stage2_sft_adapter")
model.save_pretrained(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)

# ------------------------------------------------------------------------------
# Step 6: Live Validation Test Block
# ------------------------------------------------------------------------------
print("\n" + "="*60)
print("🚀 Step 6: Executing Post-SFT Verification Inference Test...")
print("="*60)

FastLanguageModel.for_inference(model)

test_question = "I need a full refund. My delivery was delayed by a month and missed my event."

messages = [
    {"role": "system", "content": "You are an expert customer support assistant. Provide clear, accurate, and structured answers."},
    {"role": "user", "content": test_question}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        use_cache=True,
        temperature=0.3,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
clean_response = decoded_output.split("assistant\n")[-1].strip()

print(f"Test Customer Prompt:\n-> {test_question}\n")
print(f"SFT Trained Assistant Response:\n-> {clean_response}")
print("="*60)